In [11]:
import pandas as pd
import numpy as np
import os
import glob
from tqdm.notebook import tqdm

import warnings
warnings.filterwarnings('ignore')

from trading_library import *

In [12]:
# Model names and notations
# All models use 252-day rolling window
MODELS = {
    # Two features (sentiment_mean, sentiment_std)
    'lr': {
        'name': 'Linear Regression',
        'file': 'predictions_linear_regression_input=2.pkl',
        'col': 'lr'
    },
    'lasso': {
        'name': 'LASSO',
        'file': 'predictions_lasso_input=2.pkl',
        'col': 'lasso'
    },
    'elasticnet': {
        'name': 'Elastic Net',
        'file': 'predictions_elasticnet_input=2.pkl',
        'col': 'elasticnet'
    },
    'nn': {
        'name': 'Neural Network',
        'file': 'predictions_neural_network_input=2_layers=(8, 4, 2).pkl',
        'col': 'nn'
    },
    # All features
    'lr_all': {
        'name': 'Linear Regression (All Features)',
        'file': 'predictions_linear_regression_input=31.pkl',
        'col': 'lr_all'
    },
    'lasso_all': {
        'name': 'LASSO (All Features)',
        'file': 'predictions_lasso_input=31.pkl',
        'col': 'lasso_all'
    },
    'elasticnet_all': {
        'name': 'Elastic Net (All Features)',
        'file': 'predictions_elasticnet_input=31.pkl',
        'col': 'elasticnet_all'
    },
    'nn_all': {
        'name': 'Neural Network (All Features)',
        'file': 'predictions_neural_network_input=31_layers=(8, 4, 2).pkl',
        'col': 'nn_all'
    }
}

In [13]:
# Directories
PROJECT = "C:/Users/skazempour/Dropbox/Projects/42 - Machine learning from the crowd/"
DATA = os.path.join(PROJECT, "Data")
FIGURES = os.path.join(PROJECT, "Figures")
CRSP_FOLDER = "D:/CRSP"

# Input files
INPUT_DATA = os.path.join(DATA, "merged_master.pkl")

# Global parameters
START_DATE = "2012-01-01"
END_DATE = "2022-12-31"
ROLLING_WINDOW = 252

# Load and merge predictions

In [14]:
# Load the original aggregated tweets data
master_data = pd.read_pickle(INPUT_DATA)
master_data = master_data[master_data['date'] >= START_DATE]
df = master_data.copy()

# Load predictions for each model
for model_key, model_info in tqdm(MODELS.items(), desc="Loading predictions"):
    pred_file = os.path.join(DATA, model_info['file'])
    if os.path.exists(pred_file):
        pred_df = pd.read_pickle(pred_file)
        
        # Drop unnecessary columns
        cols_to_drop = [c for c in ['index', 'ticker'] if c in pred_df.columns]
        if cols_to_drop:
            pred_df = pred_df.drop(columns=cols_to_drop)
        
        # Find the rolling window prediction column
        if 'prediction' in pred_df.columns:
            # Keep only date, permno, and rolling prediction
            pred_df = pred_df[['date', 'permno', 'prediction']]
            pred_df.columns = ['date', 'permno', model_info['col']]
            df = pd.merge(df, pred_df, on=['date', 'permno'], how='inner')
            print(f"  Loaded {model_info['name']}: {len(pred_df)} predictions")
        else:
            print(f"  Warning: No prediction column found in {model_info['file']}")
    else:
        print(f"  Warning: File not found - {model_info['file']}")

print(f"\nFinal dataset: {len(df)} observations")

Loading predictions:   0%|          | 0/8 [00:00<?, ?it/s]

  Loaded Linear Regression: 14545760 predictions
  Loaded LASSO: 14545760 predictions
  Loaded Elastic Net: 14545760 predictions
  Loaded Neural Network: 14545760 predictions
  Loaded Linear Regression (All Features): 14545760 predictions
  Loaded LASSO (All Features): 14545760 predictions
  Loaded Elastic Net (All Features): 14545760 predictions
  Loaded Neural Network (All Features): 14545760 predictions

Final dataset: 14545760 observations


# Load CRSP daily data

In [15]:
# Find all dsf_final_*.pkl files
crsp_files = sorted(glob.glob(os.path.join(CRSP_FOLDER, "dsf_final_*.pkl")))
print(f"Found {len(crsp_files)} CRSP files from {crsp_files[0]} to {crsp_files[-1]}.")

# Columns to keep
cols_to_keep = ['permno', 'date', 'cap', 'ret']

# Load and combine all files
crsp = [pd.read_pickle(file)[cols_to_keep] for file in tqdm(crsp_files)]

# Combine all dataframes
crsp = pd.concat(crsp, ignore_index=True)

# Clean the data
crsp['f_ret'] = crsp.groupby('permno')['ret'].shift(-1)
crsp['date'] = pd.to_datetime(crsp['date'])

Found 17 CRSP files from D:/CRSP\dsf_final_2008.pkl to D:/CRSP\dsf_final_2024.pkl.


  0%|          | 0/17 [00:00<?, ?it/s]

# Get Fama-French factors

In [16]:
# Download Fama-French 5 factors and momentum factor
import pandas_datareader.data as web

print("Downloading Fama-French 5 factors...", end='')
ff5 = web.DataReader('F-F_Research_Data_5_Factors_2x3_daily', 'famafrench', start=START_DATE, end=END_DATE)[0]
ff5.index = pd.to_datetime(ff5.index, format='%Y%m%d')
ff5 = ff5 / 100  # Convert from percentage to decimal
print("Done!")

print("Downloading momentum factor...", end='')
mom = web.DataReader('F-F_Momentum_Factor_daily', 'famafrench', start=START_DATE, end=END_DATE)[0]
mom.index = pd.to_datetime(mom.index, format='%Y%m%d')
mom = mom / 100  # Convert from percentage to decimal
print("Done!")

# Merge FF5 and momentum into a single dataframe
ff_factors = pd.merge(ff5, mom, left_index=True, right_index=True, how='inner')

In [17]:
all_dates_to_use = np.sort(ff_factors.reset_index()['Date'].unique())
all_dates_to_use = all_dates_to_use[(all_dates_to_use >= pd.Timestamp(START_DATE)) & (all_dates_to_use <= pd.Timestamp(END_DATE))]

# Calculate portfolio returns

In [18]:
# List of prediction columns from the models dictionary
prediction_columns = [model_info['col'] for model_info in MODELS.values()]

# Dictionary to store portfolio returns for each prediction
portfolios = {}

# Calculate portfolio returns for each prediction signal
for pred_col in tqdm(prediction_columns, desc="Calculating portfolio returns"):
    if pred_col not in df.columns:
        print(f"  Skipping {pred_col} - column not found")
        continue
    
    # Form portfolios using the prediction as signal
    portfolio = form_portfolio_from_signals_sort(
        signals=df,
        returns=crsp,
        stock_col='permno',
        date_col='date',
        signal_col=pred_col,
        return_col='f_ret',
        bins=10,
        portfolio_weights=None,
        line_up_with=all_dates_to_use,
        shift=1
    )
    
    # Store in dictionary
    portfolios[pred_col] = portfolio

Calculating portfolio returns:   0%|          | 0/8 [00:00<?, ?it/s]

In [19]:
# Calculate portfolio returns with minimum stock requirements

# Define minimum stock thresholds
min_stocks_thresholds = [4, 10]

# Dictionary to store filtered portfolio returns
portfolios_filtered = {threshold: {} for threshold in min_stocks_thresholds}

for threshold in min_stocks_thresholds:
    print(f"\n{'='*80}")
    print(f"Calculating returns with minimum {threshold} stocks requirement")
    print(f"{'='*80}\n")
    
    for pred_col in prediction_columns:
        if pred_col not in portfolios:
            continue
            
        portfolio = portfolios[pred_col].copy()
        
        # Create filtered returns - set to 0 if fewer than threshold stocks
        portfolio_filtered = portfolio.copy()
        
        # For long side: set return to 0 if n_long < threshold
        portfolio_filtered.loc[portfolio['n_long'] < threshold, 'long_ret'] = 0
        portfolio_filtered.loc[portfolio['n_long'].isna(), 'long_ret'] = 0
        
        # For short side: set return to 0 if n_short < threshold
        portfolio_filtered.loc[portfolio['n_short'] < threshold, 'short_ret'] = 0
        portfolio_filtered.loc[portfolio['n_short'].isna(), 'short_ret'] = 0
        
        # Store filtered portfolio
        portfolios_filtered[threshold][pred_col] = portfolio_filtered
        
    # Calculate and display final cumulative returns
    print(f"\nFinal Log Cumulative Returns (minimum {threshold} stocks):")
    print("-" * 80)
    for pred_col in prediction_columns:
        if pred_col not in portfolios_filtered[threshold]:
            continue
            
        portfolio_filtered = portfolios_filtered[threshold][pred_col]
        
        # Calculate long-short returns
        ls_ret = portfolio_filtered['long_ret'].fillna(0) - portfolio_filtered['short_ret'].fillna(0)
        
        # Calculate log cumulative returns
        log_cum_ret = np.log(1 + ls_ret).cumsum()
        final_value = log_cum_ret.iloc[-1] if len(log_cum_ret) > 0 else 0
        
        # Count trading days
        n_trading_days = (ls_ret != 0).sum()
        
        # Get model name from MODELS dict
        model_name = next((m['name'] for m in MODELS.values() if m['col'] == pred_col), pred_col)
        print(f"{model_name:45s}: {final_value:>8.4f} ({np.exp(final_value)-1:>7.2%}) | Trading days: {n_trading_days:>4d}")

print(f"\n{'='*80}")
print("Portfolio calculation complete!")
print(f"{'='*80}")


Calculating returns with minimum 4 stocks requirement


Final Log Cumulative Returns (minimum 4 stocks):
--------------------------------------------------------------------------------
Linear Regression                            :   0.2184 ( 24.40%) | Trading days: 2767
LASSO                                        :   0.1580 ( 17.12%) | Trading days: 2199
Elastic Net                                  :   0.1576 ( 17.07%) | Trading days: 2199
Neural Network                               :  -0.1365 (-12.76%) | Trading days:  380
Linear Regression (All Features)             :   1.6838 (438.62%) | Trading days: 2767
LASSO (All Features)                         :   1.3295 (277.91%) | Trading days: 2244
Elastic Net (All Features)                   :   1.3239 (275.81%) | Trading days: 2244
Neural Network (All Features)                :   0.0249 (  2.52%) | Trading days:  145

Calculating returns with minimum 10 stocks requirement


Final Log Cumulative Returns (minimum 10 stocks):
---------

In [20]:
# Save results for use in analysis notebook
import pickle

results = {
    'portfolios': portfolios,
    'portfolios_filtered': portfolios_filtered,
    'ff_factors': ff_factors,
    'prediction_columns': prediction_columns,
    'MODELS': MODELS
}

output_file = os.path.join(DATA, 'trading_daily_results.pkl')
with open(output_file, 'wb') as f:
    pickle.dump(results, f)

print(f"Results saved to {output_file}")

Results saved to C:/Users/skazempour/Dropbox/Projects/42 - Machine learning from the crowd/Data\trading_daily_results.pkl
